# 05b — Hybrid submission: public Raptor CoAtNet + our ensemble (our weight 0.2)

GPU T4 x2, Internet OFF. Attach the competition, `knee-wheelhouse`, the training session outputs and the public dataset `nartaa/rsna-knee-publication-swa-weights-20261007` (checkpoint verified by SHA-256). First the public Raptor inference (third_party/raptor_swa_inference.py, unmodified except for its output path) runs in its own process on Kaggle's default packages; then our ensemble runs as in 05_submit; finally both are rank-blended per finding with our ensemble at weight 0.2. If the Raptor member fails, the submission is our ensemble alone.

In [ ]:
# 1. Public Raptor CoAtNet member first, in its own process, before any package is installed here.
import os, sys, subprocess, time
NOTEBOOK_START = time.time()
OWN_WEIGHT = 0.2   # rank weight of our ensemble in the final blend, fixed before scoring
RAPTOR_OUT = '/kaggle/working/raptor_submission.csv'
RAPTOR_SOURCE = '# Third-party component: unmodified except for its output path (env RAPTOR_OUT).\n# Source: Kaggle notebook nartaa/rsna-knee-0949-anatomical-mirror ("RSNA Knee: accuracy\n# inference (0.949 public AUC)"), pulled 2026-10-08 through the public Kaggle API. It runs\n# the checkpoint raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt from the dataset\n# nartaa/rsna-knee-publication-swa-weights-20261007 (CoAtNet-RMLP-2 @384, SWA), built on\n# dreaddevelopment\'s Raptor CoAtNet/MIL inference recipe. Its training recipe is not\n# published, so under the project policy its trained weights may be used as a blend member.\n# Credit and acknowledgement as required by the original notebook belong to its authors.\n\n# ---- checkpoint verification (original notebook cell 3) ----\n# Authorized scoring run: verify the selected checkpoint before inference.\nRUN_INFERENCE = True\nif not RUN_INFERENCE:\n    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")\n\nimport hashlib\nfrom pathlib import Path\nimport torch\nif not torch.cuda.is_available():\n    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")\n_weight_name = \'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt\'\n_candidates = [\n    Path(\'/kaggle/input/rsna-knee-publication-swa-weights-20261007\') / _weight_name,\n    Path(\'/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007\') / _weight_name,\n    Path(\'/kaggle/input/rsna-knee-oai-original384-s42-pair\') / _weight_name,\n    Path(\'/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair\') / _weight_name,\n]\n_weights = next((p for p in _candidates if p.is_file()), None)\nif _weights is None:\n    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")\nwith _weights.open(\'rb\') as _f:\n    assert hashlib.file_digest(_f, \'sha256\').hexdigest() == \'7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef\', "Checkpoint hash differs"\n\n\n# ---- inference (original notebook cell 4) ----\n# %% [code]\n#!/usr/bin/env python3\n"""Knee MRI: twelve findings from a single model\n\nThis notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,\nmedial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral\ncompartments, joint effusion, synovitis, a Baker\'s cyst, bone contusion and fracture. It scores\n0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.\n\nThis is the inference half of the work. The model was trained separately and its weights are\nattached as a dataset, so this notebook only loads them and predicts:\nhttps://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense\n\nWhere the training labels came from\n\nWorth saying up front, because it shapes everything else. The competition gives you 4,407 studies\nbut structured labels for only 58 of them. Every other study arrives with a free-text radiology\nreport and nothing more, so there is very little to train against out of the box.\n\nThe labels behind these weights were made by reading those reports with a language model and\nturning each into twelve probabilities rather than twelve yes or no answers. A report that says a\ntear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every\nhedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came\nwith real labels were never trained on and are used to check the result honestly; the model reaches\n0.9167 macro-AUC on them.\n\nBuilding a fixed input from studies that are all shaped differently\n\nThe hard part of this competition is not the network, it is that no two studies look alike. A\nstudy holds several DICOM series shot in different planes, the number of series varies, and the\nnumber of slices in a series varies more. Anything that expects a fixed-size input has to be given\none.\n\nThe approach here is to fill five fixed slots per study, always in the same order, for a stack of\n64 images:\n\n  18 slices from a sagittal series, preferring a fluid-sensitive one\n  14 slices from a second sagittal series, preferring one that is not fluid-sensitive\n  12 slices from a coronal series, preferring a fluid-sensitive one\n   8 slices from a second coronal series\n  12 slices from an axial series\n\nPreferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-\nsensitive sequences show swelling, effusion and acute injury clearly, while the other sequences\nshow anatomy and cartilage better, and the twelve findings are split across both. If a study has\nno series for a slot, the slot is left as zeros and the model is told to skip it rather than being\nfed something misleading.\n\nWithin a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the\nmiddle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and\ncutting them was measurably costing accuracy on exactly those findings.\n\nEvery slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from\nthe DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count\nmatters: it means a knee occupies the same fraction of the frame whether the scan was acquired at\n0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical\ninformation.\n\nHow the model reads the stack\n\nThree neighbouring slices are stacked into the three channels of one image. The network then sees\na little of what lies above and below the slice in the middle, which is most of the benefit of a 3D\nmodel at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet\nbackbone at 384 pixels.\n\nThe windows are combined with an attention layer that has separate weights for each of the twelve\nfindings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices\nwhile osteoarthritis is spread across many coronal ones, and a single pooled score forces those two\nto share one notion of which slices are important. Giving each finding its own attention weights\nlets each one draw on the slices that actually show it.\n\nRunning it\n\nScoring uses 42 windows per study. Inference runs in half precision and automatically retries a\nstudy in full precision if it fails, so no study is ever dropped from the submission. The notebook\nneeds no internet: the backbone is loaded from the attached weights rather than downloaded.\n"""\nimport os, sys, glob, time, json, gc\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\nos.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")\nimport numpy as np\nfrom concurrent.futures import ThreadPoolExecutor\nimport threading, queue\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\n# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes\n# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for\n# the fixed (1,24,3,res,res) input.\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\n\n# ---- fixed config (must match training exactly) -----------------------------\nINPUT_RES = 320\nINPUT_MODE = "crop"\nIMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample\nCROP_MM = 140.0\n# 96 slices per study at native 384 px. Must match the corpus the weights were trained on\n# (cache/build_corpus96.py).\nSLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),\n         ("Coronal", 0, 12), ("Axial", -1, 18)]\nMAXS = sum(s[2] for s in SLOTS)                     # 96\nK_EVAL = 94   # every window position of the 96-slice test volume (full coverage)\nHDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound\nPX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices\nQUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)\nNORM = "imagenet"\nFASTREAD_VARIANT = \'study_lut\'   # kernel v3: research/fastread/readers.py, byte-identical to build_study\nFR_HDR, FR_PX = 32, 16\n_FASTREAD_B64 = "..."\ndef _load_fastread():\n    import base64 as _b, importlib.util as _iu\n    p = \'/kaggle/working/_fastread.py\' if os.path.isdir(\'/kaggle/working\') else os.path.join(os.path.dirname(os.path.abspath(__file__)), \'_fastread.py\')\n    if not os.path.exists(p):\n        open(p, \'wb\').write(_b.b64decode(_FASTREAD_B64))\n    spec = _iu.spec_from_file_location(\'_fastread\', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)\n    return m\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n       "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n\n# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.\n# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm\n# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.\n# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.\n# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).\n# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),\n# cnlarge384 (0.8752), maxvit384 (0.8438).\n#\n# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.\n#\n# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while\n# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed\n# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.\n#\n# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45\n# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).\n#\n# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):\n#   incumbent CoAtNet (3,155-study corpus) 0.8923\n#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)\n# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,\n# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.\nARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean\n    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},\n]\n\n\n# ============================================================================\n# Model -- verbatim from finetune_raptor.py\n# ============================================================================\ndef build_backbone(arch, pretrained=False):\n    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable\n    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them\n    # down the ViT path (mirrors finetune_raptor.py exactly).\n    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))\n    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)\n    if is_vit:\n        kw.update(global_pool="token", dynamic_img_size=True)\n    else:\n        kw.update(global_pool="avg")\n    return timm.create_model(arch, **kw)\n\n\nclass RaptorClassifier(nn.Module):\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        # Bound backbone memory; the attention head still sees every window together.\n        flat = x.flatten(0, 1)\n        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    import hashlib\n    with open(pt_path, "rb") as stream:\n        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()\n    assert observed_sha == \'7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef\', "Bound checkpoint bytes differ"\n    print(f"[weights] sha256 {observed_sha}", flush=True)\n    ck = torch.load(pt_path, map_location="cpu", weights_only=False)\n    arch = ck.get("arch", arch_default)\n    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"\n    ck_res = INPUT_RES\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck["model"], strict=True)\n    model.eval().to(device)\n    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM\n    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows\n    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.\n    del ck\n    gc.collect()\n    return model, ck_res\n\n\n# ============================================================================\n# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)\n# ============================================================================\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = int(valid.min()), int(valid.max())\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\n\nSLOT_SIZES = [26, 22, 18, 12, 18]\nSAGITTAL_SLOTS = (0, 1)\n\ndef _slot_index(D):\n    b = np.cumsum([0] + SLOT_SIZES)\n    if b[-1] != D: raise RuntimeError(f\'slot sizes sum to {b[-1]}, volume has {D} slices\')\n    s_ = np.searchsorted(b, np.arange(D), side=\'right\') - 1\n    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)\n    return s_, rel\n\ndef _mirror_tri(tri, cs, D):\n    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each\n    window\'s CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.\n    Done on uint8 BEFORE the per-channel LUT, i.e. \'mirror the knee, then normalise\'."""\n    s_, _ = _slot_index(D)\n    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)\n    return torch.where(sag, tri.flip(1), tri.flip(-1))\n\ndef infer_pair_probs(model, plain, mirrored, device):\n    assert plain.shape==mirrored.shape==(94,3,320,320)\n    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"\n    p = infer_probs(model, plain, device)\n    q = infer_probs(model, mirrored, device)\n    return 0.5 * (p + q)\n\ndef eval_windows(vol, mask, k, res, norm=NORM, mirror=False):\n    D = vol.shape[0]\n    cs = _eval_centers(mask, D, k)\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)\n        if mirror:\n            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()\n        tri = tri.astype(np.float32) / 255.0\n        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES\n        lo = (IMG - res) // 2\n        tri = tri[:, lo:lo + res, lo:lo + res]\n        t = torch.from_numpy(np.ascontiguousarray(tri))\n        if t.shape[-1] != res:\n            t = F.interpolate(t[None], size=(res, res), mode="bilinear",\n                              align_corners=False)[0]\n        wins[j] = t.numpy()\n    x = torch.from_numpy(wins)\n    if norm == "imagenet":\n        x = (x - _MEAN) / _STD\n    return x\n\n\n_WLUT = {}\ndef gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):\n    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel\n    256-entry table computed on the CPU with eval_windows\' own float32 ops, so they are bit-identical."""\n    D = vol.shape[0]\n    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])\n    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8\n    if mirror:\n        tri = _mirror_tri(tri, cs, D)\n    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES\n    lo = (IMG - res) // 2\n    tri = tri[:, :, lo:lo + res, lo:lo + res]\n    key = (str(dev), norm)\n    if key not in _WLUT:\n        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))\n        t = torch.stack([v, v, v], 0).view(3, 256, 1)\n        if norm == \'imagenet\':\n            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)\n        _WLUT[key] = t.view(3, 256).contiguous().to(dev)\n    L = _WLUT[key]\n    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values\n    if x.shape[-1] != res:\n        if norm == \'imagenet\':   # resize must happen BEFORE normalisation, as in eval_windows\n            L0 = _WLUT.setdefault((str(dev), \'none\'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))\n            x = L0[tri.long()]\n            x = F.interpolate(x, size=(res, res), mode=\'bilinear\', align_corners=False)\n            x = (x - _MEAN.to(dev)) / _STD.to(dev)\n        else:\n            x = F.interpolate(x, size=(res, res), mode=\'bilinear\', align_corners=False)\n    return x\n\n\n@torch.no_grad()\ndef infer_probs(model, xwins, device):\n    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"\n    x = xwins.unsqueeze(0).to(device)\n    with torch.autocast("cuda", dtype=torch.float16):\n        logits = model(x).float()\n        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"\n        o = torch.sigmoid(logits)\n    return o[0].cpu().numpy()\n\n\ndef rankpct(x):                                   # per-column percentile rank in [0,1]\n    order = x.argsort(0).argsort(0).astype(np.float64)\n    return order / max(1, (x.shape[0] - 1))\n\n\n# ============================================================================\n# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST\n# ============================================================================\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def _hdr(f):\n        try:\n            h = pydicom.dcmread(f, stop_before_pixels=True)\n            iop = getattr(h, \'ImageOrientationPatient\', None)\n            ipp = getattr(h, \'ImagePositionPatient\', None)\n            if iop is not None and ipp is not None and len(iop) == 6:\n                r = np.array(iop[:3], float); c = np.array(iop[3:], float)\n                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))\n            else:\n                pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n            ps = getattr(h, \'PixelSpacing\', None); ps = float(ps[0]) if ps is not None else 0.5\n            return (pos, f, ps, True)\n        except Exception:\n            return (0.0, f, 0.5, False)\n\n    def order_and_meta(sdir):\n        # Header reads are latency-bound on Kaggle\'s mounted competition data (~240k opens on the hidden\n        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.\n        fs = glob.glob(sdir + "/*.dcm")\n        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:\n            out = list(ex.map(_hdr, fs))\n        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return [(f, ps) for _, f, ps in recs], med_ps\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))\n        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n\n    return order_and_meta, read_px, mm_crop_resize\n\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    cands = [r for r in rows if r[\'Anatomical_Plane\'] == plane and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get(\'Fluid_Sensitive\', 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\ndef build_study(sid, ser_records, tsdir, reader):\n    order_and_meta, read_px, mm_crop_resize = reader\n    rows = ser_records.get(sid, [])\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()\n    for plane, fluid, k in SLOTS:\n        r = _pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            idx += k; continue\n        used.add(r[\'SeriesInstanceUID\'])\n        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r[\'SeriesInstanceUID\']}")\n        if not files:\n            idx += k; continue\n        # wide span: the collateral ligaments and lateral meniscus live in the\n        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus\n        # the weights were trained on (build_corpus80.py, SPAN).\n        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n        sel = [files[min(p, n - 1)] for p in picks]\n        def _px(fp_ps):\n            fp, ps = fp_ps\n            try: return read_px(fp), ps\n            except Exception: return None, med_ps\n        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:\n            got = list(ex.map(_px, sel))                    # order preserved\n        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]\n        valid = [a for a in arrs if a is not None]\n        if valid:\n            allpx = np.concatenate([a.ravel() for a in valid])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = 0.0, 1.0\n        for a, ps in zip(arrs, pss):\n            if idx >= MAXS: break\n            if a is None: idx += 1; continue\n            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)\n            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)\n            vol[idx] = (aw * 255).astype(np.uint8); idx += 1\n        if idx >= MAXS: break\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return vol, mask\n\n\n\n# ---- parallel preprocessing helpers (module level so they pickle under fork) ----\n_SER = None; _TSDIR = None; _READER = None\ndef _pre_init():\n    global _READER\n    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)\ndef _pre_one(sid):\n    try:\n        vol, mask = _READER(sid, _SER, _TSDIR)\n    except Exception as e:\n        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)\n        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)\n    return sid, vol, mask\n\n# ============================================================================\n# Test-root discovery + weights + main\n# ============================================================================\ndef find_test_root():\n    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",\n             "/kaggle/input/rsna-knee-abnormality-detection"]\n    for b in cands:\n        if os.path.exists(b + "/test.csv"):\n            return b\n    for d, _, f in os.walk("/kaggle/input"):\n        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):\n            return d\n    for d, _, f in os.walk("/kaggle/input"):\n        if "test.csv" in f:\n            return d\n    raise RuntimeError("no test root under /kaggle/input")\n\n\ndef find_weight_file(fname):\n    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.\n    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",\n              f"/kaggle/input/raptor-knee-arms/1/{fname}",\n              f"/kaggle/input/raptor-cnn336/{fname}"]\n    for p in direct:\n        if os.path.exists(p):\n            return p\n    for d in sorted(glob.glob("/kaggle/input/*/")):\n        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):\n        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)\n        if "competition" in d.lower() or "abnormality-detection" in d or any(\n                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):\n            continue\n        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)\n        if hits:\n            return hits[0]\n    raise RuntimeError(f"{fname} not found under /kaggle/input")\n\n\ndef main():\n    import pandas as pd\n    t0 = time.time()\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    ngpu = torch.cuda.device_count()\n    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)\n\n    ROOT = find_test_root()\n    tsdir = ROOT + "/test_series"\n    if not os.path.isdir(tsdir):\n        tsdir = ROOT + "/test_images"\n    print("test root:", ROOT, "| series dir:", tsdir, flush=True)\n\n    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)\n    test_ids = test["StudyInstanceUID"].tolist()\n    tser = pd.read_csv(ROOT + "/test_series.csv")\n    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)\n    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)\n    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}\n    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)\n\n    sub_cols = ["StudyInstanceUID"] + LAB\n    ssub = os.path.join(ROOT, "sample_submission.csv")\n    if os.path.exists(ssub):\n        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)\n\n    N = len(test_ids); A = len(ARMS)\n    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]\n    pos = {sid: i for i, sid in enumerate(test_ids)}\n\n    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,\n    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----\n    xla = None\n    try:\n        import torch_xla; import torch_xla.core.xla_model as xm\n        dev = torch_xla.device(); xla = xm\n        print("device: XLA/TPU", dev, flush=True)\n    except Exception as e:\n        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)\n    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]\n    models = {}\n    for d in devs:\n        models[d] = []\n        for a, arm in enumerate(ARMS):\n            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)\n            models[d].append((model, res))\n        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)\n    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()\n\n    def consumer(d):\n        while True:\n            item = q.get()\n            if item is None: q.task_done(); return\n            sid, vol, mask = item; i = pos[sid]\n            try:\n                for a, (model, res) in enumerate(models[d]):\n                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)\n                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"\n                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)\n                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)\n                    del xm\n            except Exception as e:\n                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)\n            with lock:\n                done_n[0] += 1\n                if done_n[0] % 200 == 0 or done_n[0] == N:\n                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)\n            q.task_done()\n\n    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]\n    for th in threads: th.start()\n\n    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----\n    import multiprocessing as mp\n    global _SER, _TSDIR, _READER\n    _SER, _TSDIR = SER, tsdir\n    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))\n    t_pre = time.time()\n    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:\n        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):\n            q.put((sid, vol, mask))\n            if (i + 1) % 200 == 0 or i + 1 == N:\n                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)\n    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)\n    for _ in devs: q.put(None)\n    q.join()\n    for th in threads: th.join()\n    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)\n    for d in devs:\n        for m, _ in models[d]: del m\n    models.clear(); gc.collect()\n\n    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).\n    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can\n    # never silently change the scale. Falls back to equal weights if none are declared.\n    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)\n    _w = _w / _w.sum()\n    print(f"[blend] weighted rank-mean w={dict(zip([a[\'file\'] for a in ARMS], _w.round(4)))}", flush=True)\n    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),\n                         axes=(0, 0))                                          # (N,12) in [0,1]\n    if not np.isfinite(ranks).all():\n        ranks[~np.isfinite(ranks)] = 0.5\n\n    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)\n    sub.insert(0, "StudyInstanceUID", test_ids)\n    sub = sub[sub_cols]\n    assert list(sub.columns) == sub_cols, "column order drift"\n    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"\n    assert np.isfinite(sub[LAB].values).all()\n    out = os.environ.get("RAPTOR_OUT", "/kaggle/working/raptor_submission.csv")\n    sub.to_csv(out, index=False)\n    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)\n    print(sub.head().to_string(index=False), flush=True)\n    print(f"DONE {time.time()-t0:.0f}s", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
with open('/kaggle/working/raptor_infer.py', 'w') as handle: handle.write(RAPTOR_SOURCE)
try:
    subprocess.run([sys.executable, '/kaggle/working/raptor_infer.py'], check=True, timeout=5 * 3600,
                   env=dict(os.environ, RAPTOR_OUT=RAPTOR_OUT))
except Exception as exc:
    print('Raptor member failed; the submission falls back to our ensemble alone:', type(exc).__name__, exc)
RAPTOR_HOURS = (time.time() - NOTEBOOK_START) / 3600
print(f'Raptor member finished after {RAPTOR_HOURS:.2f} h')


In [ ]:
# Install offline from the attached knee-wheelhouse dataset (the same wheels shard 0 saves).
import subprocess, sys
"""Bounded discovery of attached Kaggle artifacts; embedded in launch notebooks."""
import fnmatch, os
from pathlib import Path


DICOM_DIRS = ('train_series', 'test_series')   # competition image trees: never listed


def attached_files(name, root='/kaggle/input', max_depth=5):
    # Kaggle mounts datasets and notebook outputs at different depths; artifacts live in
    # folders 1-5 levels below the root. The DICOM trees are pruned: listing them took
    # ~2 min on local disk and over an hour on the TPU VM's network-mounted inputs.
    root = Path(root)
    found = set()
    for path, dirs, files in os.walk(root):
        depth = len(Path(path).relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in DICOM_DIRS] if depth < max_depth else []
        if depth >= 1:
            found.update(Path(path) / f for f in fnmatch.filter(files, name) if not f.startswith('.'))
    return sorted(found)


def one_input(name, override='', root='/kaggle/input'):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f'Configured input does not exist: {path}')
        return str(path)
    matches = attached_files(name, root)
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one intended {name} input or set its path explicitly. '
                           f'Found: {[str(p) for p in matches]}')
    print(f'{name}: {matches[0]}')
    return str(matches[0])


def fold_inputs(expected, root='/kaggle/input', per_fold=1):
    # per_fold > 1 when several training runs (e.g. DINOv2 and DINOv3 arms) are attached.
    paths = []
    for fold in expected:
        matches = attached_files(f'fold_{fold}.pt', root)
        if len(matches) != per_fold:
            raise RuntimeError(f'Attach exactly {per_fold} fold_{fold}.pt input(s) '
                               f'({"exactly one" if per_fold == 1 else "one per training run"}). '
                               f'Found: {[str(p) for p in matches]}')
        paths.extend(str(p) for p in matches)
    for path in paths: print(f'checkpoint: {path}')
    return paths


def checkpoint_inputs(root='/kaggle/input'):
    # Every attached fold_N.pt is an ensemble member; families are resolved from each
    # checkpoint's recipe, and duplicates of one family-fold are rejected by the caller.
    paths = [str(p) for p in attached_files('fold_*.pt', root)]
    if not paths:
        raise RuntimeError('Attach at least one training output containing fold_N.pt')
    for path in paths: print(f'checkpoint: {path}')
    return paths

wheels = attached_files('timm-1.0.29-*.whl')
assert len(wheels) == 1, f'Attach exactly one wheelhouse (the knee-wheelhouse dataset, not the cache too): {wheels}'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
    '--no-deps', '--find-links', str(wheels[0].parent), 'timm==1.0.29',
    'pydicom', 'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg'])

import os, gc, glob, json, time, warnings, traceback
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

COMP_DIR = next((d for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection",
                                "/kaggle/input/rsna-knee-abnormality-detection") if os.path.isdir(d)),
                "/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT_CSV  = "/kaggle/working/own_submission.csv"   # blended into submission.csv below

BATCH        = 4
NUM_WORKERS  = 4        
MAX_HOURS    = max(1.0, 8.0 - RAPTOR_HOURS - 0.3)   # the 9 h limit covers both members      
MIN_SLICES   = 8

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

# Every attached fold_N.pt is a member. Members are rank-averaged within their family
# (reader_convnext, reader_coatnet, slot_dinov2, radimagenet), then families are averaged.
CKPT_PATHS = checkpoint_inputs()
# A knee-2c-v4 member (if any) defines the decode: v3 members then read its first six slots, which are
# pixel-identical to v3. Stable sort: v4 checkpoints first, the rest in discovery order.
CKPT_PATHS = sorted(CKPT_PATHS, key=lambda p: torch.load(p, map_location='cpu', weights_only=False)['cfg']
                    .get('preprocessing', {}).get('version') != 'knee-2c-v4')
# Family weights are fixed before each scoring run and never chosen on gold. Evidence (2026-10-08):
# OOF pseudo AUC (06_oof_blend_check): reader 0.889, equal blend 0.886, 0.7/0.1/0.2 blend 0.893;
# public LB: 17 models at equal weights 0.917 < two reader folds alone 0.923. Reader only until the
# 0.7/0.1/0.2 blend is measured. Weight 0 skips a family; every attached family must be listed.
FAMILY_WEIGHTS = {'reader_convnext': 1.0, 'slot_dinov2': 0.0, 'radimagenet': 0.0}
ckpt_path = CKPT_PATHS[0]
print("Loading ckpt")
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg = ck["cfg"]
BACKBONE   = cfg["backbone"]
IMG        = cfg["img"]
N_SLICES   = cfg["n_slices"]
PLANES     = cfg["planes"]
LABEL_COLS = cfg["label_cols"]
print(f"checkpoint: epoch {ck['epoch']} | gold AUC {ck.get('auc_gold', float('nan')):.4f}")
print(f"  {BACKBONE} | {len(PLANES)} planes x {N_SLICES} slices x {IMG}px")
MODERN = cfg.get('preprocessing',{}).get('version') in ('knee-2c-v2', 'knee-2c-v3', 'knee-2c-v4', 'knee-2c-v5')
assert MODERN or PLANES == ['Sagittal','Coronal','Axial'], 'Unsupported legacy slot recipe: use its matching inference notebook or retrain on 2c v2/v3'
if MODERN:
    IMG = cfg['preprocessing']['img']   # decoded test volumes use the cache geometry; models resize
# Members are decoded once, so they must share pixels, slots and targets. Architecture,
# backbone and seed may differ: each member is rebuilt from its own checkpoint recipe.
SHARED = ('preprocessing', 'planes', 'label_cols', 'n_slices')
SLOT_PREFIX_KEYS = ('img', 'n_slices', 'crop_mm', 'slice_band', 'canon_side', 'min_slices', 'slot_rule', 'windows', 'window_size')
def reads_slot_prefix(member, reference):
    # A knee-2c-v3 member runs on knee-2c-v4 volumes: v4's first six slots are v3's slots with the same
    # geometry (pixel-identical by construction), so the member reads volumes[:, :6].
    a, b = member.get('preprocessing', {}), reference.get('preprocessing', {})
    return (a.get('version') == 'knee-2c-v3' and b.get('version') == 'knee-2c-v4'
            and all(a.get(k) == b.get(k) for k in SLOT_PREFIX_KEYS) and b['slots'][:len(a['slots'])] == a['slots']
            and reference['planes'][:len(member['planes'])] == member['planes']
            and member.get('label_cols') == reference.get('label_cols') and member.get('n_slices') == reference.get('n_slices'))
seen_members = {}
for path in CKPT_PATHS:
    other_ck = torch.load(path,map_location='cpu',weights_only=False)
    other = other_ck['cfg']
    assert all(other.get(k) == cfg.get(k) for k in SHARED) or reads_slot_prefix(other, cfg),         f'{path}: different preprocessing, slots or labels; do not combine'
    assert MODERN or other == cfg, 'Legacy checkpoints must share one recipe'
    # Members of one family may differ by seed; a family-fold-seed may appear only once.
    key = (other.get('preset', f"{other.get('arch', 'kneenet')}:{other['backbone']}"), other_ck.get('fold', path), other.get('seed'))
    assert key not in seen_members, f'{path} duplicates {seen_members.get(key)} ({key}); attach each training output once'
    seen_members[key] = path
    del other_ck
families = {}
for family, fold, seed in seen_members: families.setdefault(family, []).append(fold if seed in (None, 42) else f'{fold}(s{seed})')
print('ensemble members:', len(CKPT_PATHS), '| families and folds:', families)

## 1 · Decoder check

The compressed transfer syntaxes (JPEG Lossless, JPEG 2000) need a backend beyond
pydicom. This notebook is offline and cannot install one, so find out **now** whether
the image ships it — and if not, which fraction of test series it costs you.

In [ ]:
import pydicom, cv2
print("pydicom", pydicom.__version__, "| cv2", cv2.__version__)

backends = {}
for name in ("pylibjpeg", "pylibjpeg_libjpeg", "pylibjpeg_openjpeg", "gdcm", "PIL"):
    try:
        __import__(name); backends[name] = True
    except ImportError:
        backends[name] = False
for k, v in backends.items():
    print(f"  {k:<20} {'yes' if v else 'NO'}")

test_series = pd.read_csv(os.path.join(COMP_DIR, "test_series.csv"))
test_series["series_dir"] = [
    os.path.join(COMP_DIR, "test_series", st, se)
    for st, se in zip(test_series["StudyInstanceUID"], test_series["SeriesInstanceUID"])]

# survey transfer syntaxes and actually try to decode one slice of each
seen = {}
for d in test_series["series_dir"].head(60):
    fs = glob.glob(os.path.join(d, "*.dcm"))
    if not fs:
        continue
    try:
        ds = pydicom.dcmread(fs[0], stop_before_pixels=True)
        ts = str(ds.file_meta.TransferSyntaxUID)
        name = ds.file_meta.TransferSyntaxUID.name
    except Exception as e:
        seen.setdefault("UNREADABLE HEADER", [0, str(e)[:60]])[0] += 1
        continue
    if ts not in seen:
        try:
            pydicom.dcmread(fs[0]).pixel_array
            seen[ts] = [1, f"OK   ({name})"]
        except Exception as e:
            seen[ts] = [1, f"FAIL ({name}): {type(e).__name__}"]
    else:
        seen[ts][0] += 1

print("\ntransfer syntaxes in the example test data:")
for ts, (n, status) in sorted(seen.items(), key=lambda kv: -kv[1][0]):
    print(f"  {n:>4} series  {status}")
if any("FAIL" in s for _, s in seen.values()):
    print("\n!! Some syntaxes cannot be decoded offline. Add pylibjpeg wheels as a")
    print("   dataset and `pip install --no-index --find-links=<dir>` before scoring.")
    print("   Those studies will fall back to the prior, costing real AUC.")

## 2 · Series selection and decoding (identical to step 2)

In [ ]:
if MODERN:
    """Shared step-2c preprocessing. Embedded verbatim in cache and inference notebooks."""
    import os, glob, re, json, hashlib
    import numpy as np
    import pandas as pd
    import pydicom
    import cv2
    from concurrent.futures import ThreadPoolExecutor
    
    # Each recipe fully determines the cached pixels. v2 stays decodable so checkpoints
    # trained on it keep their exact inputs; new caches use the current version.
    RECIPES = {
        "knee-2c-v2": dict(
            img=336, n_slices=12, crop_mm=130.0, band=(0.2, 0.8), min_slices=8, canon="L",
            slot_rule="fluid_fallback",
            slots=[("SAG_FLUID", "Sagittal", True), ("COR_FLUID", "Coronal", True),
                   ("AX_FLUID", "Axial", True)]),
        # Six header-typed slots, as in the public DINOv2 members: three fat-suppressed
        # fluid-sensitive planes plus non-suppressed sagittal fluid and two T1 slots.
        # Typing is strict; an absent sequence stays absent instead of borrowing another.
        # Each slot stores 8 windows of 3 ADJACENT slices whose centres span 6-94% of the
        # stack (the public 2.5D reader layout), at 0.54 mm/px. 37 GB: built as two shards.
        "knee-2c-v3": dict(
            img=240, n_slices=24, windows=8, crop_mm=130.0, band=(0.06, 0.94), min_slices=8,
            canon="L", slot_rule="strict", n_shards=2,
            slots=[("SAG_FLUID_FS", "Sagittal", True, True),
                   ("COR_FLUID_FS", "Coronal", True, True),
                   ("AX_FLUID_FS", "Axial", True, True),
                   ("SAG_FLUID_NOFS", "Sagittal", True, False),
                   ("COR_T1", "Coronal", False, False),
                   ("SAG_T1", "Sagittal", False, False)]),
        # v3 plus two LOCAL slots (2026-10-11): the sagittal and coronal fluid-sensitive series again,
        # cropped to 90 mm (0.375 mm/px instead of 0.54) around the knee-joint centre. The centre is
        # header geometry only: left-right from the middle of the sagittal stack, anterior-posterior
        # from the middle of the coronal stack, superior-inferior from both field centres. The six
        # v3 slots are pixel-identical to v3. 49 GB: built as three shards.
        "knee-2c-v4": dict(
            img=240, n_slices=24, windows=8, crop_mm=130.0, band=(0.06, 0.94), min_slices=8,
            canon="L", slot_rule="strict", n_shards=3, local_mm=90.0,
            slots=[("SAG_FLUID_FS", "Sagittal", True, True),
                   ("COR_FLUID_FS", "Coronal", True, True),
                   ("AX_FLUID_FS", "Axial", True, True),
                   ("SAG_FLUID_NOFS", "Sagittal", True, False),
                   ("COR_T1", "Coronal", False, False),
                   ("SAG_T1", "Sagittal", False, False),
                   ("SAG_LOCAL", "Sagittal", True, True),
                   ("COR_LOCAL", "Coronal", True, True)],
            local={"SAG_LOCAL": ["SAG_FLUID_FS", "SAG_FLUID_NOFS"], "COR_LOCAL": ["COR_FLUID_FS", "COR_T1"]}),
        # v4 at higher resolution and slice density (2026-10-12): 288 px (0.45 mm/px full field, 0.31 mm/px
        # local) and 10 windows of 3 adjacent slices per series (30 slices, so less of a 30-40 slice stack is
        # skipped). Same slots, crops and anchor. 88 GB: built as five shards.
        "knee-2c-v5": dict(
            img=288, n_slices=30, windows=10, crop_mm=130.0, band=(0.06, 0.94), min_slices=8,
            canon="L", slot_rule="strict", n_shards=5, local_mm=90.0,
            slots=[("SAG_FLUID_FS", "Sagittal", True, True),
                   ("COR_FLUID_FS", "Coronal", True, True),
                   ("AX_FLUID_FS", "Axial", True, True),
                   ("SAG_FLUID_NOFS", "Sagittal", True, False),
                   ("COR_T1", "Coronal", False, False),
                   ("SAG_T1", "Sagittal", False, False),
                   ("SAG_LOCAL", "Sagittal", True, True),
                   ("COR_LOCAL", "Coronal", True, True)],
            local={"SAG_LOCAL": ["SAG_FLUID_FS", "SAG_FLUID_NOFS"], "COR_LOCAL": ["COR_FLUID_FS", "COR_T1"]}),
    }
    PREPROCESS_VERSION = "knee-2c-v5"
    _R = RECIPES[PREPROCESS_VERSION]
    IMG, N_SLICES, CROP_MM = _R["img"], _R["n_slices"], _R["crop_mm"]
    SLICE_BAND, MIN_SLICES, CANON_SIDE = _R["band"], _R["min_slices"], _R["canon"]
    SLOTS = _R["slots"]
    HDR = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
           "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
           "PixelSpacing", "ImageOrientationPatient", "ImagePositionPatient", "Rows", "Columns"]
    
    
    def read_header(series_dir):
        rec = {t: None for t in HDR}
        files = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
        rec["n_slices"] = len(files)
        for f in files[:3]:
            try:
                ds = pydicom.dcmread(f, stop_before_pixels=True)
            except Exception:
                continue
            for t in HDR:
                v = getattr(ds, t, None)
                if v is None:
                    continue
                if t == "PixelSpacing":
                    try: v = "|".join(str(float(x)) for x in v)
                    except Exception: v = None
                elif t in ("ImageOrientationPatient", "ImagePositionPatient"):
                    try: v = "|".join(str(float(x)) for x in v)
                    except Exception: v = None
                elif t == "ScanOptions":
                    v = "|".join(map(str, v)) if not isinstance(v, str) else v
                elif not isinstance(v, (int, float, str)):
                    v = str(v)
                rec[t] = v
            break
        return rec
    
    
    _SEP    = re.compile(r"[^a-z0-9]+")
    _FATSAT = re.compile(r"\b(?:fs|fatsat|fatsuppressed|spair|spir|stir|dixon|spectral)\b")
    _T1_RX, _T2_RX = re.compile(r"\bt1\b"), re.compile(r"\bt2\b")
    _PD_RX  = re.compile(r"\b(?:pd|pdw|proton)\b")
    FATSAT_OPTS = {"FS", "FSAT", "SFS", "SP", "SPIR", "SPAIR", "FATSAT"}
    # v3 slot typing. Siemens "SP" is spatial presaturation, not fat suppression, so the
    # option set is exact; descriptions add water excitation and TIRM.
    _FATSAT_V3 = re.compile(r"\b(?:fs|fatsat|fatsuppressed|fatsup|spair|spir|stir|tirm|dixon|"
                            r"spectral|we)\b|fat sat|fat sup|water excit")
    FATSAT_OPTS_V3 = {"FS", "FSAT", "FAT_SAT", "FATSAT", "SPIR", "SPAIR"}
    
    
    def annotate(df):
        desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
        desc = desc.str.lower().str.replace(_SEP, " ", regex=True)
        opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
        df["fatsat_hdr"] = desc.str.contains(_FATSAT) | opts.apply(
            lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
        df["fatsat"] = desc.str.contains(_FATSAT_V3) | opts.apply(
            lambda ts: any(t.strip() in FATSAT_OPTS_V3 for t in ts))
        tr, te = pd.to_numeric(df["RepetitionTime"], errors="coerce"), pd.to_numeric(df["EchoTime"], errors="coerce")
        gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
        t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)
        df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                        np.where(t2 & ~pdw, "T2", np.where(pdw, "PD", np.where(gre, "GRE",
                         np.where(tr < 800, "T1", np.where(te > 60, "T2",
                          np.where(tr >= 800, "PD", "UNK")))))))
        df["fluid_hdr"] = np.isin(df["weight"], ["PD", "T2"])
        return df
    
    
    def _sort_key(ds):
        # Canonical through-plane direction: the normal's sign depends on how the scanner
        # wrote ImageOrientationPatient, so force its dominant component positive. For a
        # sagittal stack that makes slices always run in +x = toward the patient's left.
        iop = getattr(ds, "ImageOrientationPatient", None)
        ipp = getattr(ds, "ImagePositionPatient", None)
        if iop is not None and ipp is not None:
            try:
                n = np.cross(np.array(iop[:3], float), np.array(iop[3:6], float))
                n = n * np.sign(n[np.argmax(np.abs(n))] or 1.0)
                return float(np.dot(np.array(ipp, float), n))
            except Exception:
                pass
        return float(getattr(ds, "InstanceNumber", 0) or 0)
    
    
    def _row_points_left(iop):
        # +column direction in patient coords. True if increasing column goes toward
        # the patient's LEFT (+x). Used to make image handedness consistent first.
        if iop is None or len(iop) < 3:
            return True
        return float(iop[0]) >= 0
    
    
    def slice_indices(n, n_slices, band, windows=None):
        # Positions read from an ordered stack of n files.
        if windows:
            # Window centres spread over the band; each window is the centre and its two
            # neighbours, so every 3-slice input keeps true through-plane adjacency.
            a, b = max(1.0, band[0] * (n - 1)), min(n - 2.0, band[1] * (n - 1))
            if b < a:
                a = b = (n - 1) / 2
            centres = np.clip(np.round(np.linspace(a, b, windows)).astype(int), 1, max(1, n - 2))
            return np.clip(np.stack([centres - 1, centres, centres + 1], 1).reshape(-1), 0, n - 1)
        lo_i, hi_i = int(band[0] * (n - 1)), int(band[1] * (n - 1))
        return (np.linspace(lo_i, hi_i, n_slices).round().astype(int) if hi_i > lo_i
                else np.full(n_slices, n // 2))
    
    
    ANCHOR_SOURCES = {'Sagittal': ('SAG_FLUID_FS', 'SAG_FLUID_NOFS', 'SAG_T1'), 'Coronal': ('COR_FLUID_FS', 'COR_T1')}
    ANCHOR_WINDOW = (0.15, 0.85)   # a projected centre outside this share of rows/columns is not trusted
    
    
    def stack_centre(series_dir):
        """Patient-coordinate (LPS) centre of a stack: the mean of its slices' field centres."""
        centres = []
        for f in sorted(glob.glob(os.path.join(series_dir, "*.dcm"))):
            try:
                ds = pydicom.dcmread(f, stop_before_pixels=True)
                ipp = np.asarray([float(x) for x in ds.ImagePositionPatient])
                iop = np.asarray([float(x) for x in ds.ImageOrientationPatient])
                py, px = (float(x) for x in ds.PixelSpacing)
                rows, cols = int(ds.Rows), int(ds.Columns)
            except Exception:
                continue
            centres.append(ipp + iop[:3] * px * (cols - 1) / 2 + iop[3:] * py * (rows - 1) / 2)
        return np.mean(centres, 0) if centres else None
    
    
    def joint_anchor(sag_centre, cor_centre):
        """Knee-joint centre (LPS) from header geometry: left-right from the sagittal stack, anterior-
        posterior from the coronal stack, superior-inferior from both field centres. None if neither."""
        if sag_centre is not None and cor_centre is not None:
            return np.array([sag_centre[0], cor_centre[1], (sag_centre[2] + cor_centre[2]) / 2])
        return sag_centre if sag_centre is not None else cor_centre
    
    
    def project_anchor(ds, anchor, shape, flip_cols):
        """(row, col) of an LPS point in this slice's pixel grid (after the column flip), or None when
        the geometry is missing or the point falls outside ANCHOR_WINDOW of the image."""
        try:
            ipp = np.asarray([float(x) for x in ds.ImagePositionPatient])
            iop = np.asarray([float(x) for x in ds.ImageOrientationPatient])
            py, px = (float(x) for x in ds.PixelSpacing)
        except Exception:
            return None
        d = np.asarray(anchor, float) - ipp
        row, col = float(d @ iop[3:]) / py, float(d @ iop[:3]) / px
        if flip_cols:
            col = shape[1] - 1 - col
        lo, hi = ANCHOR_WINDOW
        if not (lo * shape[0] <= row <= hi * shape[0] and lo * shape[1] <= col <= hi * shape[1]):
            return None
        return int(round(row)), int(round(col))
    
    
    def square_crop(a, cy, cx, hy, hx):
        """Crop [cy-hy, cy+hy) x [cx-hx, cx+hx), zero-padded where it leaves the image (no distortion)."""
        out = np.zeros((2 * hy, 2 * hx), a.dtype)
        y0, x0 = cy - hy, cx - hx
        sy0, sx0 = max(0, y0), max(0, x0)
        sy1, sx1 = min(a.shape[0], cy + hy), min(a.shape[1], cx + hx)
        if sy1 > sy0 and sx1 > sx0:
            out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = a[sy0:sy1, sx0:sx1]
        return out
    
    
    def load_series(series_dir, plane, side, n_slices=N_SLICES, img=IMG,
                    band=SLICE_BAND, crop_mm=CROP_MM, canon=CANON_SIDE, windows=None, center_lps=None):
        # Callers pass slot_kwargs(recipe, slot, anchor) so a cache or inference run reproduces its recipe.
        # center_lps (local slots): crop a zero-padded square around that patient-coordinate point when it
        # projects inside the image; otherwise, and for every regular slot, the centred crop below.
        files = sorted(glob.glob(os.path.join(series_dir, "*.dcm")))
        if not files:
            raise FileNotFoundError(series_dir)
        keyed = []
        for f in files:
            try:
                keyed.append((_sort_key(pydicom.dcmread(f, stop_before_pixels=True)), f))
            except Exception:
                continue
        if not keyed:
            raise RuntimeError("no readable headers")
        keyed.sort(key=lambda t: t[0])
        ordered = [f for _, f in keyed]
    
        # Sagittal: L-R is the through-plane axis, so a right knee is the stack reversed.
        # Reverse the FILE ORDER here, before the slice band is applied -- reversing the
        # finished volume instead would sample anatomical positions shifted by one slice
        # relative to a left knee.
        if side != canon and plane == "Sagittal":
            ordered = ordered[::-1]
    
        n = len(ordered)
        idx = slice_indices(n, n_slices, band, windows)
        if windows: assert len(idx) == n_slices == 3 * windows
    
        out = np.zeros((n_slices, img, img), dtype=np.float32)
        ok, flip_cols = 0, False
        decoded = np.zeros(n_slices, dtype=bool)
        for j, i in enumerate(idx):
            try:
                ds = pydicom.dcmread(ordered[int(i)])
                a = ds.pixel_array.astype(np.float32)
                m = float(getattr(ds, "RescaleSlope", 1) or 1)
                b = float(getattr(ds, "RescaleIntercept", 0) or 0)
                if (m, b) != (1.0, 0.0):
                    a = a * m + b
                if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
                    a = a.max() - a
                if a.ndim == 3:
                    a = a.mean(axis=-1)
    
                iop = getattr(ds, "ImageOrientationPatient", None)
                flip_cols = plane in ("Coronal", "Axial") and not _row_points_left(iop)
                if flip_cols:
                    a = a[:, ::-1]
    
                # physical crop, centred, before the resize
                ps = getattr(ds, "PixelSpacing", None)
                if ps is not None and len(ps) == 2:
                    py, px = map(float, ps)
                    if np.isfinite([py, px]).all() and min(py, px) > 0:
                        hy, hx = max(1, round(crop_mm / py / 2)), max(1, round(crop_mm / px / 2))
                        cy, cx = a.shape[0] // 2, a.shape[1] // 2
                        at = project_anchor(ds, center_lps, a.shape, flip_cols) if center_lps is not None else None
                        if at is not None:
                            a = square_crop(a, at[0], at[1], hy, hx)
                        else:
                            a = a[max(0, cy-hy):min(a.shape[0], cy+hy),
                                  max(0, cx-hx):min(a.shape[1], cx+hx)]
    
                out[j] = cv2.resize(np.ascontiguousarray(a), (img, img),
                                    interpolation=cv2.INTER_AREA)
                ok += 1
                decoded[j] = True
            except Exception:
                continue
        if ok == 0:
            raise RuntimeError("no decodable slices")
    
        # Replace failed slices by nearest decoded neighbour before normalization.
        good = np.flatnonzero(decoded)
        for j in np.flatnonzero(~decoded):
            out[j] = out[good[np.argmin(abs(good-j))]]
        lo_v, hi_v = np.percentile(out, [1, 99])
        if hi_v <= lo_v:
            lo_v, hi_v = float(out.min()), float(out.max())
        if hi_v <= lo_v:
            hi_v = lo_v + 1.0
        out = np.clip((out - lo_v) / (hi_v - lo_v) * 255.0, 0, 255).astype(np.uint8)
    
        # ---- laterality normalisation, in-plane half ----------------------------
        # +x is the patient's left and columns now point left, so for coronal/axial a
        # right knee is the column-mirror of a left knee. (The sagittal half was done
        # above, on the file order.)
        if side != canon and plane != "Sagittal":
            out = out[:, :, ::-1]
        return np.ascontiguousarray(out)
    
    def _vec(value):
        try:
            return np.asarray([float(x) for x in value.split('|')], dtype=float)
        except (AttributeError, ValueError, TypeError):
            return None
    
    
    def series_side(row):
        for tag in ('ImageLaterality', 'Laterality'):
            side = str(row.get(tag, '')).strip().upper()
            if side in ('L', 'R'):
                return side, 'tag'
        ipp, iop, ps = (_vec(row.get(k)) for k in
                        ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing'))
        try:
            rows, cols = float(row['Rows']), float(row['Columns'])
            if ipp is not None and iop is not None and ps is not None:
                center = ipp + iop[:3] * ps[1] * (cols-1)/2 + iop[3:] * ps[0] * (rows-1)/2
                if np.isfinite(center).all() and abs(center[0]) > 5:
                    return ('L' if center[0] > 0 else 'R'), 'coords'
        except (KeyError, ValueError, TypeError):
            pass
        return None, 'unknown'
    
    
    def vote(group):
        # Explicit tags outrank any number of coordinate guesses. Ties stay unknown.
        tagged = group.loc[group['side_src'].eq('tag'), 'side'].dropna()
        values = tagged if len(tagged) else group['side'].dropna()
        counts = values.value_counts()
        if not len(counts) or (len(counts)>1 and counts.iloc[0] == counts.iloc[1]):
            return CANON_SIDE, 'default'
        return counts.index[0], 'tag' if len(tagged) else 'coords'
    
    
    def recipe(version=PREPROCESS_VERSION):
        if version not in RECIPES:
            raise ValueError(f'Unknown preprocessing version {version!r}')
        return RECIPES[version]
    
    
    def series_kwargs(spec):
        # load_series arguments that reproduce one recipe's pixels exactly.
        return dict(n_slices=spec['n_slices'], img=spec['img'], band=tuple(spec['band']),
                    crop_mm=spec['crop_mm'], canon=spec['canon'], windows=spec.get('windows'))
    
    
    def slot_kwargs(spec, slot, anchor=None):
        # Per-slot load_series arguments: local slots crop local_mm around the study's joint anchor.
        kw = series_kwargs(spec)
        if spec['slots'][slot][0] in spec.get('local', {}):
            kw['crop_mm'] = spec['local_mm']
            if anchor is not None and np.isfinite(np.asarray(anchor, float)).all():
                kw['center_lps'] = np.asarray(anchor, float)
        return kw
    
    
    def row_anchor(row):
        # The manifest's joint anchor for one study, or None (absent columns or unknown geometry).
        try:
            anchor = np.array([float(row['anchor_x']), float(row['anchor_y']), float(row['anchor_z'])])
        except (KeyError, TypeError, ValueError):
            return None
        return anchor if np.isfinite(anchor).all() else None
    
    
    def choose(group, spec=None):
        spec = recipe() if spec is None else spec
        picked = {}
        local = spec.get('local', {})
        for slot in spec['slots']:
            name, plane, fluid = slot[:3]
            if name in local:
                continue
            sel = group['plane'].eq(plane) & group['fluid'].eq(fluid)
            if spec['slot_rule'] == 'strict':
                sel &= group['fatsat'].eq(slot[3])
                if not fluid:
                    # T1-typed series before any other non-fluid (GRE/unknown) series.
                    sel &= group['weight'].isin(['T1', 'GRE', 'UNK'])
                cand = group[sel & (group['n_slices'] > 0)]
                if len(cand):
                    rank = cand.assign(_t1=cand['weight'].eq('T1'),
                                       _enough=cand['n_slices'] >= spec['min_slices'])
                    picked[name] = rank.sort_values(['_enough', '_t1', 'n_slices', 'series_dir'],
                                                    ascending=[False, False, False, True]).iloc[0]
                continue
            cand = group[sel & (group['n_slices'] >= spec['min_slices'])]
            if not len(cand): cand = group[sel & (group['n_slices'] > 0)]
            if not len(cand): cand = group[group['plane'].eq(plane) & (group['n_slices'] >= spec['min_slices'])]
            if len(cand):
                picked[name] = cand.sort_values(['n_slices', 'series_dir'], ascending=[False, True]).iloc[0]
        # Local slots re-read the first available source slot's series with the local crop.
        for name, sources in local.items():
            source = next((picked[s] for s in sources if s in picked), None)
            if source is not None:
                picked[name] = source
        return picked
    
    
    def build_manifest(studies, series, spec=None):
        spec = recipe() if spec is None else spec
        slots = spec['slots']
        if series.empty:
            result = studies[['StudyInstanceUID']].copy()
            result['study_side'], result['side_src'] = spec['canon'], 'default'
            for i in range(len(slots)): result[f'dir_{i}'] = ''
            return result
        series = series.copy().reset_index(drop=True)
        with ThreadPoolExecutor(max_workers=8) as pool:
            headers = pd.DataFrame(list(pool.map(read_header, series['series_dir'])))
        # Header columns may already exist in an input CSV.
        for col in headers: series[col] = headers[col]
        series = annotate(series)
        provided = pd.to_numeric(series['Fluid_Sensitive'], errors='coerce').fillna(0).astype(bool)
        series['fluid'] = np.where(series['weight'].eq('UNK'), provided, series['fluid_hdr'])
        series['plane'] = series['Anatomical_Plane'].astype(str)
        sides = [series_side(row) for _, row in series.iterrows()]
        series['side'] = [s[0] for s in sides]
        series['side_src'] = [s[1] for s in sides]
        records = []
        for uid, group in series.groupby('StudyInstanceUID', sort=False):
            side, src = vote(group)
            rec = dict(StudyInstanceUID=uid, study_side=side, side_src=src)
            picked = choose(group, spec)
            for i, slot in enumerate(slots):
                row = picked.get(slot[0])
                rec[f'dir_{i}'] = row['series_dir'] if row is not None else ''
            if spec.get('local'):
                rec['_anchor_dirs'] = [next((picked[s]['series_dir'] for s in ANCHOR_SOURCES[plane] if s in picked), None)
                                       for plane in ('Sagittal', 'Coronal')]
            records.append(rec)
        anchor_cols = ['anchor_x', 'anchor_y', 'anchor_z'] if spec.get('local') else []
        if anchor_cols:
            # Joint anchor per study from header geometry (every slice header of one sagittal and one coronal stack).
            dirs = sorted({d for rec in records for d in rec['_anchor_dirs'] if d})
            with ThreadPoolExecutor(max_workers=8) as pool:
                centre_of = dict(zip(dirs, pool.map(stack_centre, dirs)))
            for rec in records:
                sag, cor = (centre_of.get(d) if d else None for d in rec.pop('_anchor_dirs'))
                anchor = joint_anchor(sag, cor)
                rec.update(zip(anchor_cols, anchor if anchor is not None else [np.nan] * 3))
        columns = ['StudyInstanceUID', 'study_side', 'side_src', *[f'dir_{i}' for i in range(len(slots))], *anchor_cols]
        result = studies[['StudyInstanceUID']].merge(pd.DataFrame(records, columns=columns),
                                                   on='StudyInstanceUID', how='left', validate='one_to_one')
        result['study_side'] = result['study_side'].fillna(spec['canon'])
        result['side_src'] = result['side_src'].fillna('default')
        for i in range(len(slots)): result[f'dir_{i}'] = result[f'dir_{i}'].fillna('')
        print('Laterality sources:', result['side_src'].value_counts().to_dict())
        return result
    
    
    def preprocessing_meta(version=PREPROCESS_VERSION):
        spec = recipe(version)
        meta = dict(version=version, slots=[s[0] for s in spec['slots']],
                    planes=[s[1] for s in spec['slots']], n_slices=spec['n_slices'], img=spec['img'],
                    crop_mm=spec['crop_mm'], slice_band=list(spec['band']), canon_side=spec['canon'],
                    min_slices=spec['min_slices'])
        if spec['slot_rule'] != 'fluid_fallback':
            # v2 metadata is kept byte-identical so its checkpoints still validate.
            meta.update(slot_rule=spec['slot_rule'],
                        slot_types=[[bool(s[2]), bool(s[3])] for s in spec['slots']])
        if spec.get('windows'):
            meta.update(windows=spec['windows'], window_size=3)
        if spec.get('local'):
            meta.update(local={k: list(v) for k, v in spec['local'].items()}, local_mm=spec['local_mm'],
                        anchor='header geometry: LR sagittal stack, AP coronal stack, SI both field centres')
        return meta
    
    
    def window_positions(meta):
        # Relative stack position (0..1) of each window centre, for position embeddings.
        if meta.get('windows'):
            return np.linspace(meta['slice_band'][0], meta['slice_band'][1], meta['windows']).tolist()
        return np.linspace(meta['slice_band'][0], meta['slice_band'][1], meta['n_slices']).tolist()
    
    
    def cache_fingerprint(manifest, meta):
        content = manifest.to_json(orient='split') + json.dumps(meta, sort_keys=True)
        return hashlib.sha256(content.encode()).hexdigest()
    pp = cfg['preprocessing']
    RECIPE = recipe(pp['version'])
    assert preprocessing_meta(pp['version']) == pp, 'Preprocessing recipe mismatch: rebuild this notebook from the matching source'
    SLOTS = RECIPE['slots']
    SERIES_KW = series_kwargs(RECIPE)
else:
    def _sort_key(ds):
        # Canonicalise direction: the normal's sign depends on how the scanner wrote
        # ImageOrientationPatient, so force its dominant component positive and every
        # stack runs the same anatomical way. Must match step 2 exactly.
        iop = getattr(ds, "ImageOrientationPatient", None)
        ipp = getattr(ds, "ImagePositionPatient", None)
        if iop is not None and ipp is not None:
            try:
                n = np.cross(np.array(iop[:3], float), np.array(iop[3:6], float))
                n = n * np.sign(n[np.argmax(np.abs(n))] or 1.0)
                return float(np.dot(np.array(ipp, float), n))
            except Exception:
                pass
        return float(getattr(ds, "InstanceNumber", 0) or 0)
    
    
    def load_series(series_dir, n_slices=None, img=None):
        n_slices = n_slices or N_SLICES
        img = img or IMG
        files = glob.glob(os.path.join(series_dir, "*.dcm"))
        if not files:
            raise FileNotFoundError(series_dir)
    
        keyed = []
        for f in files:
            try:
                keyed.append((_sort_key(pydicom.dcmread(f, stop_before_pixels=True)), f))
            except Exception:
                continue
        if not keyed:
            raise RuntimeError("no readable headers")
        keyed.sort(key=lambda t: t[0])
        ordered = [f for _, f in keyed]
    
        idx = np.linspace(0, len(ordered) - 1, n_slices).round().astype(int)
        out = np.zeros((n_slices, img, img), dtype=np.float32)
        ok = 0
        for j, i in enumerate(idx):
            try:
                ds = pydicom.dcmread(ordered[i])
                a = ds.pixel_array.astype(np.float32)
                m = float(getattr(ds, "RescaleSlope", 1) or 1)
                b = float(getattr(ds, "RescaleIntercept", 0) or 0)
                if (m, b) != (1.0, 0.0):
                    a = a * m + b
                if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
                    a = a.max() - a
                if a.ndim == 3:
                    a = a.mean(axis=-1)
                out[j] = cv2.resize(a, (img, img), interpolation=cv2.INTER_AREA)
                ok += 1
            except Exception:
                continue
        if ok == 0:
            raise RuntimeError("no decodable slices")
    
        # whole-volume percentiles: shift/scale invariant, unlike masking to out > 0
        lo, hi = np.percentile(out, [1, 99])
        if hi <= lo:
            lo, hi = float(out.min()), float(out.max())
        if hi <= lo:
            hi = lo + 1.0
        return np.clip((out - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)

In [ ]:
if MODERN:
    test = pd.read_csv(os.path.join(COMP_DIR,'test.csv'))
    manifest = build_manifest(test,test_series,RECIPE)
else:
    test = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
    print(f"test studies: {len(test):,}")
    
    ts = test_series.copy()
    ts["Fluid_Sensitive"] = pd.to_numeric(ts["Fluid_Sensitive"], errors="coerce").fillna(0).astype(int)
    ts["n_slices"] = [len(glob.glob(os.path.join(d, "*.dcm"))) for d in
                      tqdm(ts["series_dir"], desc="counting slices", unit="series")]
    
    def choose(g):
        picked = {}
        for plane in PLANES:
            cand = g[(g["Anatomical_Plane"] == plane) & (g["n_slices"] >= MIN_SLICES)]
            if not len(cand):
                cand = g[g["Anatomical_Plane"] == plane]
            if len(cand):
                picked[plane] = cand.sort_values(["Fluid_Sensitive", "n_slices"],
                                                 ascending=[False, False]).iloc[0]
        return picked
    
    rows = []
    for uid, g in ts.groupby("StudyInstanceUID", sort=False):
        picked = choose(g)
        rec = {"StudyInstanceUID": uid}
        for i, plane in enumerate(PLANES):
            s = picked.get(plane)
            rec[f"dir_{i}"] = s["series_dir"] if s is not None else ""
        rows.append(rec)
    
    manifest = test[["StudyInstanceUID"]].merge(pd.DataFrame(rows),
                                                on="StudyInstanceUID", how="left")
    for i in range(len(PLANES)):
        manifest[f"dir_{i}"] = manifest[f"dir_{i}"].fillna("")
    
    for i, plane in enumerate(PLANES):
        print(f"  {plane:<9} available for {(manifest[f'dir_{i}'] != '').mean():6.1%} of studies")
    print("studies with no series at all:",
          int((manifest[[f"dir_{i}" for i in range(len(PLANES))]] == "").all(axis=1).sum()))

## 3 · Model

In [ ]:
"""Shared model architecture for training and offline inference."""
import numpy as np
import torch
import torch.nn as nn
import timm

def make_neighbours(x):
    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    # Attention over slices within one plane.
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):               # x (B,T,D)
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    # Per-label attention over planes: each finding picks the view it needs.
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        a = self.score(x)                          # (B,S,L)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)                       # over planes, per label
        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)
        return z, a


class KneeNet(nn.Module):
    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,
                 pretrained=True, drop=0.3, img=224, normalization="legacy",
                 encoder_chunk=64):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.head_w, std=0.02)

    def forward(self, x, present):
        # autocast entered HERE, not in the training loop: it is thread-local, so a
        # loop-level autocast never reaches a threaded forward. Verified reaching the
        # encoder as float16.
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div(255.)   # ImageNet grey stats
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)

            x3 = (x3 - self.input_mean) / self.input_std
            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.
            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:
                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])
            else:
                features = self.enc(x3)
            f = features.reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)

            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            z, plane_att = self.plane_pool(v, m)
            z = self.drop(z)
            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, plane_att



def slice_thirds(n):
    # Lower, middle and upper thirds of the cached slice band.
    parts = np.array_split(np.arange(n), 3)
    if min(len(p) for p in parts) == 0:
        return [np.array([0]), np.array([(n - 1) // 2]), np.array([n - 1])]
    return parts


def spread_groups(n):
    # 3-slice inputs that each span the band: one slice from every third. For six cached
    # slices that is [0, 2, 4] and [1, 3, 5]; a 3-slice input is its own single group.
    parts = slice_thirds(n)
    return [[int(p[g]) for p in parts] for g in range(min(len(p) for p in parts))]


class SlotQueryHead(nn.Module):
    # Per-finding query attention over series slots (public DINOv2 member head).
    def __init__(self, dim, n_slots, n_out, hidden=256, drop=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        h = self.proj(x) + self.slot_emb
        a = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        a = a.masked_fill(~mask.unsqueeze(1), torch.finfo(a.dtype).min).softmax(-1)
        z = self.drop(torch.einsum("bos,bsh->boh", a, h))
        return (z * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias, a


class SlotQueryNet(nn.Module):
    """One 3-slice image per series slot, encoder partly fine-tuned, query head on top.

    Training feeds one spread triplet per slot (T=3). Evaluation feeds every cached slice
    and averages logits over the spread groups, so no slice position is left unseen.
    """
    def __init__(self, backbone="vit_small_patch14_dinov2.lvd142m", n_out=12, n_slots=6,
                 pretrained=True, img=336, drop=0.2, hidden=256, unfreeze_last=6,
                 normalization="pretrained", encoder_chunk=48):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.img = img
        self.encoder_chunk = encoder_chunk
        self.tokens = hasattr(self.enc, "blocks") and hasattr(self.enc, "num_prefix_tokens")
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        if unfreeze_last is not None and hasattr(self.enc, "blocks"):
            # Early blocks stay at their self-supervised weights; only late blocks adapt.
            for p in self.enc.parameters(): p.requires_grad = False
            blocks = list(self.enc.blocks)
            for block in blocks[max(0, len(blocks) - unfreeze_last):]:
                for p in block.parameters(): p.requires_grad = True
            for name in ("norm", "fc_norm"):
                module = getattr(self.enc, name, None)
                if isinstance(module, nn.Module):
                    for p in module.parameters(): p.requires_grad = True
        dim = self.enc.num_features * 2
        self.head = SlotQueryHead(dim, n_slots, n_out, hidden=hidden, drop=drop)

    def encode(self, x):                           # (N,3,H,W) normalised -> (N,2D)
        f = self.enc.forward_features(x)
        if self.tokens:
            # CLS token plus mean of patch tokens; register tokens are excluded.
            return torch.cat([f[:, 0], f[:, self.enc.num_prefix_tokens:].mean(1)], dim=1)
        if f.ndim == 4:
            return torch.cat([f.mean((-2, -1)), f.amax((-2, -1))], dim=1)
        return torch.cat([f, f], dim=1)

    def forward(self, x, present):
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            keep = m.reshape(-1)
            groups = spread_groups(T)
            logits, attention = 0., 0.
            for group in groups:
                images = x[:, :, group].reshape(B * S, 3, H, W)[keep].float().div(255.)
                if H != self.img:
                    images = nn.functional.interpolate(images, size=(self.img, self.img),
                                                       mode="bilinear", align_corners=False)
                images = (images - self.input_mean) / self.input_std
                # Absent slots are never encoded; their features stay zero and masked.
                if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
                    f = torch.cat([self.encode(part) for part in images.split(self.encoder_chunk)])
                else:
                    f = self.encode(images)
                features = f.new_zeros(B * S, f.shape[-1])
                features[keep] = f
                z, a = self.head(features.reshape(B, S, -1), m)
                logits, attention = logits + z, attention + a
        return logits / len(groups), attention / len(groups)


class Reader25D(nn.Module):
    """2.5D study reader (public ConvNeXt reader layout).

    Every cached window of three adjacent slices is one CNN image. Its feature becomes a
    token tagged with its series slot and its position along the stack; a small
    transformer mixes all tokens of the study and each finding pools them with its own
    attention. Training augments each series with one shared rigid/intensity jitter (no
    flips: laterality is already normalised) and encodes a random subset of windows.
    """
    def __init__(self, backbone="convnext_tiny.in12k_ft_in1k", n_out=12, n_slots=6,
                 pretrained=True, img=240, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 normalization="pretrained", band=(0.06, 0.94), train_windows=6,
                 encoder_chunk=96, slot_dropout=0.0, aux_dim=0):
        super().__init__()
        self.slot_dropout = slot_dropout     # training only: hide whole series (protocols vary by site)
        # Runtime switches for XLA/TPU training; neither changes the parameters or the checkpoint.
        # static_shapes: encode a fixed number of windows for every slot (absent ones masked), so
        # tensor shapes never depend on the data. augment_in_model=False: the data loader already
        # applied the per-series jitter (augment_series_cpu), so the model only normalises.
        self.static_shapes = False
        self.augment_in_model = True
        # Test-time window subsets (eval only; inference WINDOW_TTA): encode a random subset of this many
        # windows per series, as in training, instead of all of them. None: every window.
        self.subset_windows = None
        kwargs = {"drop_path_rate": drop_path} if drop_path else {}
        if backbone.startswith("vit_"):
            kwargs["img_size"] = img
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kwargs)
        self.img, self.band, self.train_windows = img, tuple(band), train_windows
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim),
                                  nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(n_slots, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True,
                                           norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n_out))
        self.cls_w = nn.Parameter(torch.randn(n_out, dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        # Report-grounded auxiliary head (training only): regresses the study's report embedding
        # from the finding-pooled and the mean study tokens. Its output is kept in last_aux; the
        # logits and the inference path do not depend on it.
        self.aux = (nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Linear(dim, aux_dim))
                    if aux_dim else None)
        self.last_aux = None

    def _aux(self, h, pad, pooled):
        if self.aux is None:
            return
        keep = (~pad).unsqueeze(-1).to(h.dtype)
        mean = (h * keep).sum(1) / keep.sum(1).clamp_min(1)
        self.last_aux = self.aux(torch.cat([pooled.mean(1), mean], -1))

    def views(self, x, series, n_series):
        # x float [N,3,H,W] in 0..1; one random transform per series in training.
        if x.shape[-1] != self.img:
            x = nn.functional.interpolate(x, size=(self.img, self.img), mode="bilinear",
                                          align_corners=False, antialias=x.shape[-1] > self.img)
        if self.training and self.augment_in_model:
            dev = x.device
            ang = (torch.rand(n_series, device=dev) - .5) * 2 * np.radians(12)
            sc = 1 + (torch.rand(n_series, device=dev) - .5) * .24
            tx, ty = ((torch.rand(2, n_series, device=dev) - .5) * .12)
            theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),
                                 torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)
            grid = nn.functional.affine_grid(theta[series], x.shape, align_corners=False)
            x = nn.functional.grid_sample(x, grid, mode="bilinear", padding_mode="zeros",
                                          align_corners=False)
            gamma = torch.exp((torch.rand(n_series, device=dev) - .5) * .5)[series]
            gain = (1 + (torch.rand(n_series, device=dev) - .5) * .2)[series]
            x = x.clamp_min(1e-4).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)
        return (x - self.input_mean) / self.input_std

    def forward(self, x, present):
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            K = T // 3
            m = present.clone()
            if self.training and self.slot_dropout > 0:
                kept = m & (torch.rand(m.shape, device=m.device) >= self.slot_dropout)
                # A study that lost every series keeps one of its present series, chosen at random.
                pick = torch.rand(m.shape, device=m.device).masked_fill(~m, -1).argmax(dim=1)
                lost = ~kept.any(dim=1) & m.any(dim=1)
                one = nn.functional.one_hot(pick, m.shape[1]).bool()
                m = kept | (lost.unsqueeze(1) & one)   # fixed-shape boolean algebra (XLA-safe)
            m = m | ~m.any(dim=1, keepdim=True)    # guard: never all-masked
            if self.static_shapes:
                return self._forward_static(x, m, B, S, K, H, W)
            tokens = m.unsqueeze(-1).expand(B, S, K).clone()
            n_win = self.train_windows if self.training else getattr(self, 'subset_windows', None)
            if n_win and n_win < K:
                # Encode a random subset of windows per series: stack augmentation that
                # also bounds activation memory. Positions stay exact.
                rank = torch.rand(B, S, K, device=x.device).argsort(-1).argsort(-1)
                tokens &= rank < n_win
            keep = tokens.reshape(-1)
            series = torch.arange(B * S, device=x.device).repeat_interleave(K)[keep]
            images = x[:, :, :3 * K].reshape(B * S * K, 3, H, W)[keep].float().div(255.)
            images = self.views(images, series, B * S).contiguous(memory_format=torch.channels_last)
            if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
                f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])
            else:
                f = self.enc(images)
            feat = f.new_zeros(B * S * K, f.shape[-1])
            feat[keep] = f
            slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, K)
            pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)
            h = (self.proj(feat).view(B, S, K, -1) + self.slot_emb(slot)
                 + self.pos_emb(pos.view(1, 1, K, 1).to(feat.dtype)))
            h = h.reshape(B, S * K, -1)
            pad = ~tokens.reshape(B, S * K)
            if self.tf is not None:
                h = self.tf(h, src_key_padding_mask=pad)
            a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
            pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
            logits = (pooled * self.cls_w).sum(-1) + self.cls_b
            self._aux(h, pad, pooled)
        return logits, a


def _reader_forward_static(self, x, m, B, S, K, H, W):
    """Reader25D forward with data-independent shapes (XLA/TPU). Every slot encodes the same
    number of windows; absent or dropped slots are encoded too but masked, exactly as the
    dynamic path masks them, so outputs agree up to floating-point error."""
    n_win = self.train_windows if self.training else getattr(self, 'subset_windows', None)
    n_win = n_win if n_win and n_win < K else K
    if n_win < K:
        idx = torch.rand(B, S, K, device=x.device).argsort(-1)[..., :n_win].sort(-1).values
    else:
        idx = torch.arange(K, device=x.device).view(1, 1, K).expand(B, S, K)
    windows = x[:, :, :3 * K].reshape(B, S, K, 3, H, W)
    windows = torch.gather(windows, 2, idx.reshape(B, S, n_win, 1, 1, 1).expand(B, S, n_win, 3, H, W))
    images = windows.reshape(B * S * n_win, 3, H, W).float().div(255.)
    series = torch.arange(B * S, device=x.device).repeat_interleave(n_win)
    images = self.views(images, series, B * S)
    if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
        f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])
    else:
        f = self.enc(images)
    tokens = m.unsqueeze(-1).expand(B, S, n_win)
    slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, n_win)
    pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)[idx]
    h = (self.proj(f).view(B, S, n_win, -1) + self.slot_emb(slot)
         + self.pos_emb(pos.unsqueeze(-1).to(f.dtype)))
    h = h.reshape(B, S * n_win, -1)
    pad = ~tokens.reshape(B, S * n_win)
    if self.tf is not None:
        h = self.tf(h, src_key_padding_mask=pad)
    a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
    pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
    logits = (pooled * self.cls_w).sum(-1) + self.cls_b
    self._aux(h, pad, pooled)
    return logits, a


Reader25D._forward_static = _reader_forward_static


def augment_series_cpu(x, generator=None):
    """Reader25D's training jitter on the CPU, for loaders that feed XLA devices: one rotation
    (+-12 deg), scale (+-12%), shift (+-6%), gamma and gain per series, shared by all its slices.
    x: uint8 [S, T, H, W] of one study; returns uint8 of the same shape (values clipped to 0..255)."""
    S, T, H, W = x.shape
    rnd = lambda *shape: torch.rand(*shape, generator=generator)
    ang = (rnd(S) - .5) * 2 * np.radians(12)
    sc = 1 + (rnd(S) - .5) * .24
    tx, ty = (rnd(2, S) - .5) * .12
    theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),
                         torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)
    xf = x.float().div(255.)
    grid = nn.functional.affine_grid(theta, (S, T, H, W), align_corners=False)
    xf = nn.functional.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    gamma = torch.exp((rnd(S) - .5) * .5).view(S, 1, 1, 1)
    gain = (1 + (rnd(S) - .5) * .2).view(S, 1, 1, 1)
    xf = xf.clamp_min(1e-4).pow(gamma) * gain
    return (xf.clamp(0, 1) * 255).round().to(torch.uint8)


def window_centres(n_slices):
    # Centre slice of each cached 3-slice window (v3 layout); plain slices otherwise.
    return list(range(1, n_slices, 3)) if n_slices % 3 == 0 and n_slices >= 6 else list(range(n_slices))


class RadImageNetHead(nn.Module):
    """Frozen official RadImageNet ResNet-50 slice encoder + per-finding query head.

    Follows the public RadImageNet arm (E9/E11): every window-centre slice is one
    grey image scaled to [-1, 1]; its 2048-d global-average feature is a token tagged
    with slot and stack position; one query per finding cross-attends over all tokens
    and is fused with the mean token. Only the head trains.
    """
    def __init__(self, n_out=12, n_slots=6, n_windows=8, img=224, head_dim=512,
                 weights_path=None, encoder_chunk=128):
        super().__init__()
        from torchvision.models import resnet50
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
        if weights_path:
            state = torch.load(weights_path, map_location="cpu", weights_only=True)
            self.backbone.load_state_dict({k.split("backbone.", 1)[1]: v for k, v in state.items()},
                                          strict=True)
        for p in self.backbone.parameters(): p.requires_grad = False
        self.img, self.n_windows, self.encoder_chunk = img, n_windows, encoder_chunk
        dim = 2048
        self.project = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, head_dim), nn.GELU())
        self.plane = nn.Parameter(torch.randn(n_slots, head_dim) * .01)
        self.position = nn.Parameter(torch.randn(n_windows, head_dim) * .01)
        self.query = nn.Parameter(torch.randn(n_out, head_dim) * .02)
        self.attn = nn.MultiheadAttention(head_dim, 8, dropout=.10, batch_first=True)
        self.fuse = nn.Sequential(nn.LayerNorm(head_dim * 4), nn.Linear(head_dim * 4, head_dim),
                                  nn.GELU(), nn.Dropout(.15))
        self.weight = nn.Parameter(torch.randn(n_out, head_dim) * .02)
        self.bias = nn.Parameter(torch.zeros(n_out))

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()                       # frozen BatchNorm statistics
        return self

    @torch.no_grad()
    def encode(self, x, present):
        # x uint8 (B,S,T,H,W) -> tokens (B, S*K, 2048) float16, token mask (B, S*K)
        B, S, T, H, W = x.shape
        centres = window_centres(T)[:self.n_windows]
        K = len(centres)
        keep = present.unsqueeze(-1).expand(B, S, K).reshape(-1)
        images = x[:, :, centres].reshape(B * S * K, 1, H, W)[keep].float().div(127.5).sub(1.)
        if H != self.img:
            images = nn.functional.interpolate(images, size=(self.img, self.img), mode="bilinear",
                                               align_corners=False)
        images = images.expand(-1, 3, -1, -1)
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            f = torch.cat([self.backbone(part).mean(dim=(2, 3)) for part in images.split(self.encoder_chunk)]) \
                if len(images) else images.new_zeros(0, 2048)
        tokens = torch.zeros(B * S * K, 2048, device=x.device, dtype=torch.float16)
        tokens[keep] = f.to(torch.float16)
        return tokens.view(B, S * K, 2048), keep.view(B, S * K).float()

    def head(self, feature, mask):
        n_slots = self.plane.shape[0]
        token = self.project(feature.float())
        token = token.view(len(token), n_slots, -1, token.shape[-1])
        token = token + self.plane[None, :, None] + self.position[None, None, :token.shape[2]]
        token = token.flatten(1, 2)
        pad = mask <= 0
        empty = pad.all(1)
        if empty.any():
            pad = pad.clone(); pad[empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=pad, need_weights=False)[0]
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = mean.expand(-1, len(self.query), -1)
        fused = self.fuse(torch.cat([attended, mean, (attended - mean).abs(), attended * mean], -1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias

    def forward(self, x, present):
        m = present.clone()
        m[~m.any(dim=1)] = True
        return self.head(*self.encode(x, m)), None


def build_model(cfg, n_out, n_planes, pretrained=True):
    # A checkpoint's recipe rebuilds its own architecture; old recipes have no arch key.
    arch = cfg.get('arch', 'kneenet')
    if arch == 'kneenet':
        return KneeNet(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                       drop=cfg.get('drop', .3), img=cfg['img'],
                       normalization=cfg.get('normalization', 'legacy'))
    if arch == 'slot_query':
        return SlotQueryNet(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                            img=cfg['img'], drop=cfg.get('drop', .2),
                            hidden=cfg.get('head_hidden', 256),
                            unfreeze_last=cfg.get('unfreeze_last', 6),
                            normalization=cfg.get('normalization', 'pretrained'))
    if arch == 'reader25d':
        return Reader25D(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                         img=cfg['img'], drop=cfg.get('drop', .2),
                         dim=cfg.get('head_hidden', 384), drop_path=cfg.get('drop_path', .1),
                         band=tuple(cfg['preprocessing']['slice_band']),
                         train_windows=cfg.get('train_windows', 6),
                         normalization=cfg.get('normalization', 'pretrained'),
                         slot_dropout=cfg.get('slot_dropout', 0.0), aux_dim=cfg.get('aux_dim', 0))
    if arch == 'radhead':
        # The frozen encoder ships inside every checkpoint; weights_path is training-only.
        return RadImageNetHead(n_out, n_planes, n_windows=cfg.get('n_windows', 8), img=cfg['img'],
                               head_dim=cfg.get('head_hidden', 512),
                               weights_path=cfg.get('weights_path') if pretrained else None)
    raise ValueError(f'Unknown architecture {arch!r}')

def rank_blend(predictions):
    return np.mean([pd.DataFrame(p).rank(pct=True,method='average').to_numpy()
                    for p in predictions],axis=0)

## 4 · Inference

In [ ]:
"""Decode once, run fold models sequentially, rank-ensemble, and write a submission."""
class TestDS(Dataset):
    def __init__(self, manifest): self.manifest = manifest
    def __len__(self): return len(self.manifest)
    def __getitem__(self, i):
        row = self.manifest.iloc[i]
        vol = np.zeros((len(PLANES), N_SLICES, IMG, IMG), np.uint8)
        present = np.zeros(len(PLANES), bool)
        for slot in range(len(PLANES)):
            directory = row[f'dir_{slot}']
            if not directory: continue
            try:
                vol[slot] = (load_series(directory, SLOTS[slot][1], row['study_side'],
                                         **slot_kwargs(RECIPE, slot, row_anchor(row)))
                             if MODERN else load_series(directory))
                present[slot] = True
            except Exception:
                continue
        return torch.from_numpy(vol), torch.from_numpy(present), i

# A scratch disk cache avoids repeating expensive DICOM decoding for every fold.
# It is private to this run and deleted once the CSV has been written.
import tempfile, shutil
# The disk with the most free space holds the decoded test volumes (v3: ~8 MB per study).
_scratch_roots = [d for d in ('/kaggle/tmp', tempfile.gettempdir(), os.path.dirname(OUT_CSV)) if os.path.isdir(d)]
scratch = tempfile.mkdtemp(prefix='knee-test-', dir=max(_scratch_roots, key=lambda d: shutil.disk_usage(d).free))
shape = (len(manifest), len(PLANES), N_SLICES, IMG, IMG)
needed = int(np.prod(shape))
assert shutil.disk_usage(scratch).free > needed + 1024**3, 'Insufficient scratch space for decode-once inference'
volumes = np.lib.format.open_memmap(os.path.join(scratch,'test.npy'), mode='w+', dtype=np.uint8, shape=shape)
presence = np.zeros(shape[:2], bool)
started = time.time()
# Single-process loading isolates DICOM failures and avoids worker crashes aborting the iterator.
for i in tqdm(range(len(manifest)), desc='decode once'):
    if time.time()-started > MAX_HOURS*3600*.65:
        print('Decode budget reached; remaining rows use fallback'); break
    try:
        x, present, _ = TestDS(manifest)[i]
        volumes[i] = x.numpy(); presence[i] = present.numpy()
    except Exception as exc:
        print(f'Study {i}: {type(exc).__name__}')
volumes.flush()
all_predictions, member_families, member_paths = [], [], []
FAMILY_WEIGHTS_GIVEN = globals().get('FAMILY_WEIGHTS') or {}
WINDOW_TTA = int(globals().get('WINDOW_TTA', 0))   # extra passes over random training-size window subsets
coverage = []
for checkpoint in CKPT_PATHS:
    if time.time()-started > MAX_HOURS*3600: break
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)
    member = f"{state['cfg'].get('arch', 'kneenet')}:{state['cfg']['backbone']}"
    family = state['cfg'].get('preset', member)
    if FAMILY_WEIGHTS_GIVEN and FAMILY_WEIGHTS_GIVEN.get(family, 1.0) == 0:
        # A zero-weight family cannot change the blend; skip its inference time.
        coverage.append(dict(checkpoint=checkpoint, member=member, family=family, predicted=0, skipped='weight 0'))
        del state
        continue
    # Members may differ in architecture or backbone; each is rebuilt from its own recipe. A v3 member on
    # v4 volumes reads the first len(its planes) slots (checked when the members were listed).
    n_slots = len(state['cfg'].get('planes', PLANES))
    model = build_model(state['cfg'], len(LABEL_COLS), n_slots, pretrained=False).to(DEV)
    model.load_state_dict(state['model'], strict=True)
    del state
    model.eval()
    pred = np.full((len(manifest),len(LABEL_COLS)), np.nan, np.float32)
    valid_rows = np.flatnonzero(presence.any(1))
    def predict_rows(rows):
        if not len(rows): return
        try:
            x = torch.from_numpy(np.asarray(volumes[rows][:, :n_slots]).copy()).to(DEV)
            p = torch.from_numpy(presence[rows][:, :n_slots].copy()).to(DEV)
            with torch.inference_mode():
                logits, _ = model(x,p)
                values = logits.float().sigmoid()
                if WINDOW_TTA and getattr(model, 'train_windows', None):
                    # Half the full pass, half the mean of WINDOW_TTA passes over random training-size window subsets.
                    model.subset_windows = model.train_windows
                    subsets = torch.stack([model(x,p)[0].float().sigmoid() for _ in range(WINDOW_TTA)]).mean(0)
                    model.subset_windows = None
                    values = .5 * values + .5 * subsets
                values = values.cpu().numpy()
            finite = np.isfinite(values).all(1)
            pred[rows[finite]] = values[finite]
        except Exception as exc:
            if DEV=='cuda': torch.cuda.empty_cache()
            if len(rows)>1:
                middle = len(rows)//2
                predict_rows(rows[:middle]); predict_rows(rows[middle:])
            else: print(f'Inference failed for row {rows[0]}: {type(exc).__name__}')
    for offset in tqdm(range(0,len(valid_rows),BATCH),desc=os.path.basename(checkpoint)):
        if time.time()-started > MAX_HOURS*3600: break
        predict_rows(valid_rows[offset:offset+BATCH])
    good = np.isfinite(pred).all(1)
    coverage.append(dict(checkpoint=checkpoint, member=member, family=family, predicted=int(good.sum())))
    # A partially evaluated member would distort ranks for its unprocessed tail; omit it.
    if good.sum() == len(valid_rows) and good.any():
        pred[~good] = np.median(pred[good], 0)
        all_predictions.append(pred); member_families.append(family); member_paths.append(checkpoint)
    elif not all_predictions and good.any():
        pred[~good] = np.median(pred[good], 0)
        all_predictions.append(pred); member_families.append(family); member_paths.append(checkpoint)
    del model
    gc.collect()
    if DEV=='cuda': torch.cuda.empty_cache()

# ---- optional transductive self-training on the test set (TTST, ours, 2026-10-09) ----------------
# Test studies come from many sites and protocols. Each reader member is fine-tuned briefly on the
# test volumes themselves, with the weighted ensemble's mean probability as soft targets and its
# usual training augmentations (affine/intensity jitter, window subsets, slot dropout), then
# predicts again. No label of any kind is used. Off unless TTST_EPOCHS > 0; a member that does not
# finish inside the time budget keeps its first-pass prediction.
TTST_EPOCHS = int(globals().get('TTST_EPOCHS', 0))
TTST_LR = float(globals().get('TTST_LR', 1e-4))     # head rate; the encoder trains at 0.2x
TTST_BATCH = int(globals().get('TTST_BATCH', 4))
ttst_log = []
if TTST_EPOCHS > 0 and len(all_predictions) > 0:
    _fw = globals().get('FAMILY_WEIGHTS') or {}
    _use = [k for k, f in enumerate(member_families) if _fw.get(f, 1.0) > 0]
    soft_target = np.mean([all_predictions[k] for k in _use], axis=0).astype(np.float32)
    rows_all = np.flatnonzero(presence.any(1))
    ttst_rng = np.random.default_rng(0)
    for k in _use:
        if time.time() - started > MAX_HOURS * 3600 * .9: break
        state = torch.load(member_paths[k], map_location='cpu', weights_only=False)
        if state['cfg'].get('arch') != 'reader25d' or len(state['cfg'].get('planes', PLANES)) != len(PLANES):
            del state; continue          # TTST fine-tunes full-slot readers only
        model = build_model(state['cfg'], len(LABEL_COLS), len(PLANES), pretrained=False).to(DEV)
        model.load_state_dict(state['model'], strict=True)
        del state
        enc_params = list(model.enc.parameters())
        enc_ids = {id(q) for q in enc_params}
        opt = torch.optim.AdamW([{'params': enc_params, 'lr': TTST_LR * .2},
                                 {'params': [q for q in model.parameters() if id(q) not in enc_ids], 'lr': TTST_LR}],
                                weight_decay=.02)
        scaler = torch.amp.GradScaler('cuda', enabled=DEV == 'cuda')
        finished, losses = True, []
        for _ in range(TTST_EPOCHS):
            model.train()
            order = rows_all[ttst_rng.permutation(len(rows_all))]
            for off in range(0, len(order), TTST_BATCH):
                if time.time() - started > MAX_HOURS * 3600 * .9:
                    finished = False; break
                b = np.sort(order[off:off + TTST_BATCH])
                x = torch.from_numpy(np.asarray(volumes[b]).copy()).to(DEV)
                p = torch.from_numpy(presence[b]).to(DEV)
                y = torch.from_numpy(soft_target[b]).to(DEV)
                logits, _ = model(x, p)
                loss = nn.functional.binary_cross_entropy_with_logits(logits.float(), y)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
                scaler.step(opt); scaler.update()
                losses.append(float(loss.detach()))
            if not finished: break
        if finished:
            model.eval()
            pred = np.full((len(manifest), len(LABEL_COLS)), np.nan, np.float32)
            with torch.inference_mode():
                for off in range(0, len(rows_all), BATCH):
                    b = rows_all[off:off + BATCH]
                    logits, _ = model(torch.from_numpy(np.asarray(volumes[b]).copy()).to(DEV),
                                      torch.from_numpy(presence[b]).to(DEV))
                    pred[b] = logits.float().sigmoid().cpu().numpy()
            good = np.isfinite(pred).all(1)
            if good[rows_all].all():
                pred[~good] = np.median(pred[good], 0)
                all_predictions[k] = pred
            else:
                finished = False
        ttst_log.append(dict(member=member_paths[k], finished=bool(finished), steps=len(losses),
                             first_loss=losses[0] if losses else None, last_loss=losses[-1] if losses else None))
        print('TTST', ttst_log[-1])
        del model, opt
        gc.collect()
        if DEV == 'cuda': torch.cuda.empty_cache()

# Rank-average within each family, then across families, so a family's vote does not
# grow with its number of folds. FAMILY_WEIGHTS (default equal) is fixed before scoring, from OOF evidence.
family_weights = globals().get('FAMILY_WEIGHTS') or {}
# Explicit weights must name every attached family: a new family would otherwise get weight 1.
unweighted = sorted(set(member_families) - set(family_weights)) if family_weights else []
assert not unweighted, f'FAMILY_WEIGHTS has no weight for {unweighted}; choose one with 06_oof_blend_check'
family_ranks = {}
for fam in dict.fromkeys(member_families):
    family_ranks[fam] = rank_blend([p for p, f in zip(all_predictions, member_families) if f == fam])
if not all_predictions:
    preds = np.full((len(manifest),len(LABEL_COLS)), .5)
elif len(all_predictions) == 1:
    preds = all_predictions[0]
else:
    total = sum(family_weights.get(f, 1.0) for f in family_ranks)
    preds = sum(family_weights.get(f, 1.0) * r for f, r in family_ranks.items()) / total
# Undecoded studies take the median, not the mean: with rare findings the mean ranks above most studies.
fill = np.median(preds, 0) if len(preds) else np.full(len(LABEL_COLS),.5)
sub = pd.DataFrame(preds,columns=LABEL_COLS)
sub.insert(0,'StudyInstanceUID',manifest['StudyInstanceUID'].to_numpy())
sample = pd.read_csv(os.path.join(COMP_DIR,'sample_submission.csv'))
sub = sample[['StudyInstanceUID']].merge(sub,on='StudyInstanceUID',how='left',validate='one_to_one')
for k,col in enumerate(LABEL_COLS): sub[col] = sub[col].fillna(float(fill[k]))
sub = sub[sample.columns]
assert len(sub)==len(sample) and np.isfinite(sub[LABEL_COLS].to_numpy()).all()
assert sub[LABEL_COLS].to_numpy().min()>=0 and sub[LABEL_COLS].to_numpy().max()<=1
sub.to_csv(OUT_CSV,index=False)
diagnostics = dict(models=coverage, ensemble_members=len(all_predictions),
                   families={f: member_families.count(f) for f in family_ranks},
                   family_weights={f: family_weights.get(f, 1.0) for f in family_ranks},
                   blank_studies=int((~presence.any(1)).sum()), ttst=ttst_log,
                   largest_tie={c:int(sub[c].value_counts().max()) for c in LABEL_COLS},
                   elapsed_minutes=(time.time()-started)/60)
# Studies with no decoded series: a downstream blend keeps the other model's rank for them.
diagnostics['blank_study_ids'] = manifest.loc[~presence.any(1), 'StudyInstanceUID'].astype(str).tolist()
with open(os.path.join(os.path.dirname(OUT_CSV),'inference_diagnostics.json'),'w') as handle:
    json.dump(diagnostics,handle,indent=2)
print('Wrote',OUT_CSV,diagnostics)
del volumes
shutil.rmtree(scratch)

In [ ]:
# Fallback coverage and tie diagnostics are written with submission.csv above.

## 5 · Write the submission

In [ ]:
print(pd.read_csv(OUT_CSV).head().to_string(index=False))

In [ ]:
# 3. Rank blend per finding: (1 - OWN_WEIGHT) x Raptor + OWN_WEIGHT x our ensemble.
import numpy as np, pandas as pd
own = pd.read_csv(OUT_CSV, dtype={'StudyInstanceUID': str})
labels = [c for c in own.columns if c != 'StudyInstanceUID']
final, used = own.copy(), 'our ensemble alone'
if os.path.exists(RAPTOR_OUT):
    rap = pd.read_csv(RAPTOR_OUT, dtype={'StudyInstanceUID': str})
    good = (set(rap.StudyInstanceUID) == set(own.StudyInstanceUID) and len(rap) == len(own)
            and list(rap.columns) == list(own.columns) and np.isfinite(rap[labels].to_numpy(float)).all())
    if good:
        rap = rap.set_index('StudyInstanceUID').loc[own.StudyInstanceUID].reset_index()
        for c in labels:
            final[c] = (1 - OWN_WEIGHT) * rap[c].rank(pct=True).to_numpy() + OWN_WEIGHT * own[c].rank(pct=True).to_numpy()
        used = f'Raptor x {1 - OWN_WEIGHT:.2f} + our ensemble x {OWN_WEIGHT:.2f}'
    else:
        print('Raptor output does not match the test rows or columns; using our ensemble alone')
assert final[labels].notna().all().all() and len(final) == len(own)
final.to_csv('/kaggle/working/submission.csv', index=False)
print('submission.csv:', used, final.shape)
